# 🚀 Fine-Tune MATLAB Error-Fixer (0.5B–0.8B) with Unsloth on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/S-SaiTarun/matlab-fixer/blob/main/notebooks/train_matlab_fixer.ipynb)

This notebook fine-tunes a compact, highly capable **0.5B–0.8B code LLM** (`unsloth/Qwen2.5-Coder-0.5B-Instruct` or `Qwen/Qwen3.5-0.8B`) specifically to detect, explain, and fix common MATLAB programming bugs:
- **Matrix vs Element-Wise Operators** (`*` vs `.*`, `^` vs `.^`, `/` vs `./`, `\` vs `.\`)
- **Missing `end` Keywords** (loops, conditional blocks, function definitions, switch statements)
- **1-Based Indexing & Cell Arrays** (`0`-based/negative indices, `{}` vs `()` confusion)
- **Dimension Mismatches** (inner dimension requirements, matrix concatenation `[A, B]` vs `[A; B]`)
- **Valid MATLAB Verification** (~15% bug-free snippets where the model correctly outputs "No errors found")
- **Built-in Shadowing & Syntax Pitfalls** (shadowed functions like `sum`, `mean`, unquoted chars, `=` in `if`)

---
### ⚡ Hardware & Resource Profile
- **Target Hardware:** Free Google Colab **T4 GPU** (~15 GB VRAM)
- **VRAM Usage:** ~2.1 GB VRAM (via 4-bit QLoRA with Unsloth)
- **Training Time:** ~3–5 minutes for 3 epochs on 500 examples
- **Export Output:** Merged **4-bit `q4_k_m` GGUF** saved directly to Google Drive (`/content/drive/MyDrive/matlab-fixer-model/`)

## 1. Verify GPU Environment

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Install Unsloth and Dependencies (Colab Optimized)

In [ ]:
# Fast install for Google Colab
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes triton
!pip install datasets sentencepiece protobuf

## 3. Clone Repository & Sync Dataset (`data/matlab_train.jsonl`)

This step clones your GitHub repository directly into `/content/matlab-fixer` so the dataset is loaded automatically.

In [ ]:
import os

# Update with your repository URL
REPO_URL = "https://github.com/S-SaiTarun/matlab-fixer.git"
REPO_DIR = "/content/matlab-fixer"

# Optional: If your repository is private, you can add GITHUB_TOKEN to Colab Secrets (Left sidebar > Key icon)
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = os.environ.get('GITHUB_TOKEN')

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} into {REPO_DIR}...")
    if gh_token:
        auth_url = REPO_URL.replace("https://", f"https://{gh_token}@")
        !git clone {auth_url} {REPO_DIR}
    else:
        !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Pulling latest changes in {REPO_DIR}...")
    !cd {REPO_DIR} && git pull

DATASET_PATH = f"{REPO_DIR}/data/matlab_train.jsonl"
print(f"Dataset located at: {DATASET_PATH}")
assert os.path.exists(DATASET_PATH), f"Dataset file not found at {DATASET_PATH}!"

## 4. Download and Load Base Model in 4-bit via Unsloth

We use `unsloth/Qwen2.5-Coder-0.5B-Instruct` (or `Qwen/Qwen3.5-0.8B`), pre-quantized in 4-bit mode. This ensures all model weights download directly inside the Google Colab VM without touching your local machine.

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024  # Fits MATLAB functions, diagnostic reasoning, and fixes comfortably
dtype = None           # Auto-detection (Float16 on T4)
load_in_4bit = True    # 4-bit quantization for minimal VRAM footprint

# Model choices: 'unsloth/Qwen2.5-Coder-0.5B-Instruct' or 'unsloth/Qwen2.5-Coder-1.5B-Instruct'
model_name = "unsloth/Qwen2.5-Coder-0.5B-Instruct"

print(f"Loading {model_name} with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("Model and tokenizer successfully loaded!")

## 5. Configure LoRA Fine-Tuning Adapters

We attach LoRA adapters to all projection layers with hyperparameters:
- `r = 16`
- `lora_alpha = 16`
- `lora_dropout = 0` (optimized by Unsloth for speed)
- `target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]`

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Saves 30% VRAM
    random_state=3407,
)

trainable_params, all_params = model.get_nb_trainable_parameters()
print(f"Trainable LoRA parameters: {trainable_params:,} ({100 * trainable_params / all_params:.2f}% of {all_params:,} total)")

## 6. Format Dataset with ChatML Template

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Set ChatML template for Qwen-Coder
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
train_dataset = raw_dataset.map(formatting_prompts_func, batched=True)

print(f"Total training examples: {len(train_dataset)}")
print("\n--- Sample ChatML Formatted Entry ---")
print(train_dataset[0]["text"][:500] + "...")

## 7. Fine-Tune with SFTTrainer (3 Epochs)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Starting LoRA fine-tuning...")
trainer_stats = trainer.train()
print("Training completed successfully!")

## 8. Test Fine-Tuned Model Inference

We test our fine-tuned MATLAB model on both buggy and clean snippets to verify its diagnostic ability.

In [ ]:
FastLanguageModel.for_inference(model)

test_cases = [
    # Test 1: Element-wise operator error
    "t = 0:0.01:1;\ny = sin(2*pi*5*t) * cos(2*pi*2*t);\nplot(t, y);",
    # Test 2: Missing end
    "total = 0;\nfor k = 1:10\n    total = total + k^2;\ndisp(total);",
    # Test 3: Valid code snippet (should say No errors found)
    "x = linspace(-2, 2, 200);\ny = exp(-x.^2);\narea = trapz(x, y);"
]

for i, test_code in enumerate(test_cases, 1):
    messages = [
        {"role": "system", "content": "You are an expert MATLAB debugger and code fixer. Analyze the provided MATLAB code, identify any syntax, semantic, or runtime errors, explain the root cause, and provide the corrected code. If the code is already valid and bug-free, explain why it is correct and state that no errors were found."},
        {"role": "user", "content": f"The following MATLAB code has an issue or needs verification:\n\n```matlab\n{test_code}\n```"}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True, temperature=0.2)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    print(f"\n==================== TEST CASE #{i} ====================")
    print("Input Code:\n", test_code)
    print("\nModel Diagnosis & Fix:\n", response.strip())

## 9. Mount Google Drive and Export Merged 4-bit `q4_k_m` GGUF

This mounts your Google Drive and exports the merged model directly as a **4-bit `q4_k_m` GGUF file** into `/content/drive/MyDrive/matlab-fixer-model/` for use with `ollama`, `llama.cpp`, or LM Studio.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/matlab-fixer-model"
os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Export destination: {GDRIVE_OUTPUT_DIR}")

# Save merged 4-bit q4_k_m GGUF directly to Google Drive
print("Converting and saving merged model to 4-bit q4_k_m GGUF format...")
model.save_pretrained_gguf(
    GDRIVE_OUTPUT_DIR,
    tokenizer,
    quantization_method="q4_k_m"
)

print(f"\n[SUCCESS] GGUF model exported to Google Drive!")
!ls -lh "{GDRIVE_OUTPUT_DIR}"

## 10. (Optional) Push Adapter & GGUF to Hugging Face Hub

If `HF_TOKEN` is configured in Colab Secrets or the environment, this cell pushes the LoRA adapter and the 4-bit GGUF model directly to Hugging Face Hub.

In [ ]:
import os

# Retrieve HF_TOKEN from Colab Secrets or environment
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    HF_REPO = "S-SaiTarun/matlab-fixer-0.5B-GGUF"  # Change to your target HF repo name
    print(f"Pushing LoRA adapter and q4_k_m GGUF to Hugging Face: {HF_REPO}...")
    
    # Push 16-bit LoRA adapter
    model.push_to_hub_merged(HF_REPO + "-adapter", tokenizer, save_method="lora", token=hf_token)
    
    # Push 4-bit q4_k_m GGUF
    model.push_to_hub_gguf(HF_REPO, tokenizer, quantization_method="q4_k_m", token=hf_token)
    print(f"[SUCCESS] Published to https://huggingface.co/{HF_REPO}")
else:
    print("HF_TOKEN not found in Colab Secrets or environment.")
    print("To publish to Hugging Face Hub, add your token with write access to Colab Secrets as 'HF_TOKEN'.")